# Data Science – Zusammenfassung Woche 1

Diese Zusammenfassung basiert auf den Vorlesungsunterlagen und den zugehörigen Repositories aus den ZIP-Dateien.
Die Codebeispiele orientieren sich am Stil der vorhandenen Jupyter Notebooks und verwenden die dort enthaltenen Datensätze.

> **Fokus Woche 1:** Nur relationale Datenbanken und SQL. Themen wie NoSQL, MQTT oder Apache Kafka sind bewusst nicht enthalten.

---

# Woche 1 – SQL und relationale Daten

## 1. Datengrundlage: Northwind

Das Repository `python_postgresql_pgadmin` verwendet die Northwind-Datenbank. Die Tabellen werden aus den vorhandenen CSV-Dateien aufgebaut.

Wichtige Dateien:

| CSV-Datei | Inhalt |
|---|---|
| `Customers.csv` | Kunden |
| `Orders.csv` | Bestellungen |
| `OrderDetails.csv` | Bestellpositionen |
| `Products.csv` | Produkte |
| `Categories.csv` | Produktkategorien |
| `Employees.csv` | Mitarbeitende |
| `Shippers.csv` | Versandunternehmen |
| `Suppliers.csv` | Lieferanten |

Beispielhafte Beziehungen:


```text
Customers
   |
   | CustomerID
   v
Orders
   |
   | OrderID
   v
OrderDetails
   |
   | ProductID
   v
Products
```

Zusätzlich referenziert `Orders` die Tabellen `Employees` und `Shippers`, während `Products` mit `Categories` und `Suppliers` verbunden ist.

## 2. Primary Key und Foreign Key

Ein **Primary Key** identifiziert einen Datensatz eindeutig.

Beispiel:


```sql
CustomerID
```

Ein **Foreign Key** verweist auf den Primary Key einer anderen Tabelle und stellt damit eine Beziehung zwischen Tabellen her.

Beispiel:


```text
Customers.CustomerID
        ↑
        |
Orders.CustomerID
```

Typische Beziehung im Northwind-Datensatz:


```text
1 Customer -> n Orders
```

Ein Kunde kann also mehrere Bestellungen besitzen.

---

## 3. SELECT – Daten abfragen

Grundstruktur:


```sql
SELECT Spalte1, Spalte2
FROM Tabelle;
```

Alle Spalten:


```sql
SELECT *
FROM Customers;
```

Nur bestimmte Spalten:


```sql
SELECT
    CustomerName,
    Country
FROM Customers;
```

### Beispiel aus den Übungen

Alle Kunden aus Spanien:


```sql
SELECT * 
FROM Customers
WHERE Country = 'Spain';
```

### Merke

`SELECT *` ist zum Erkunden praktisch. Wenn nur bestimmte Informationen benötigt werden, sollten die Spalten explizit angegeben werden.

---

## 4. WHERE – Daten filtern

Mit `WHERE` werden einzelne Datensätze gefiltert.


```sql
SELECT *
FROM Products
WHERE Price > 50;
```

Wichtige Vergleichsoperatoren:

| Operator | Bedeutung |
|---|---|
| `=` | gleich |
| `<>` | ungleich |
| `>` | grösser als |
| `<` | kleiner als |
| `>=` | grösser oder gleich |
| `<=` | kleiner oder gleich |

Mehrere Bedingungen:


```sql
SELECT *
FROM Customers
WHERE Country = 'Germany'
  AND City = 'Berlin';
```

### Pitfall: AND und OR

`AND` hat eine höhere Priorität als `OR`. Bei gemischten Bedingungen sind Klammern sicherer.


```sql
WHERE (Country = 'Germany' OR Country = 'Switzerland')
  AND City IS NOT NULL;
```

---

## 5. DISTINCT und LIKE

### DISTINCT

`DISTINCT` entfernt doppelte Werte aus dem Resultat.


```sql
SELECT DISTINCT Country
FROM Customers;
```

### LIKE

`LIKE` wird für Textmuster verwendet.

| Wildcard | Bedeutung |
|---|---|
| `%` | beliebig viele Zeichen |
| `_` | genau ein Zeichen |

Beim Erzeugen des SQL-Strings mit %-Formatierung muss ein Prozentzeichen als `%%` geschrieben werden, damit ein einzelnes `%` erhalten bleibt. Darum steht im Beispiel `LIKE '%%B%%'`; im ausgeführten SQL entspricht das `LIKE '%B%'` und sucht nach einem `B` an beliebiger Stelle.

### Beispiel aus den Übungen

Eindeutige deutsche Städte, deren Name ein `B` enthält:


```sql
SELECT DISTINCT City 
FROM Customers 
WHERE Country = 'Germany' 
AND City LIKE '%%B%%';
```

---

## 6. NULL

`NULL` bedeutet, dass kein beziehungsweise kein bekannter Wert vorhanden ist.

Falsch:


```sql
WHERE CategoryID = NULL;
```

Richtig:


```sql
WHERE CategoryID IS NULL;
```

### Beispiel aus den Übungen

Produkte ohne zugewiesene Kategorie:


```sql
SELECT 
    ProductID, 
    ProductName,
    CategoryID
FROM Products
WHERE CategoryID IS NULL;
```

### Merke


```text
NULL != 0
NULL != ''
NULL != false
```

---

## 7. ORDER BY und LIMIT

### Sortieren


```sql
SELECT *
FROM Products
ORDER BY Price DESC;
```

`ASC` = aufsteigend  
`DESC` = absteigend

### Ergebnis begrenzen

PostgreSQL verwendet `LIMIT`:


```sql
SELECT *
FROM Products
ORDER BY Price DESC
LIMIT 5;
```

### Beispiel aus den Übungen

Top 5 der teuersten Produkte:


```sql
SELECT 
    ProductName, 
    ROUND(CAST(Price AS NUMERIC), 2) AS Price
FROM Products
ORDER BY Price DESC
LIMIT 5;
```

### Pitfall

Ohne `ORDER BY` ist die Reihenfolge eines SQL-Resultats nicht garantiert.

---

## 8. BETWEEN

`BETWEEN` prüft einen Wertebereich inklusive der Grenzen.


```sql
SELECT 
    ProductID, 
    ProductName, 
    ROUND(CAST(Price AS NUMERIC), 2) AS Price
FROM Products
WHERE Price BETWEEN 10 AND 50
ORDER BY Price DESC;
```

Bedeutung:


```text
10 <= Price <= 50
```

---

## 9. Aggregatfunktionen

Aggregatfunktionen fassen mehrere Datensätze zu einem Ergebnis zusammen.

| Funktion | Bedeutung |
|---|---|
| `COUNT()` | Anzahl |
| `SUM()` | Summe |
| `AVG()` | Mittelwert |
| `MIN()` | Minimum |
| `MAX()` | Maximum |

### Beispiel aus den Übungen


```sql
SELECT 
    ROUND(CAST(AVG(Price) AS NUMERIC), 2) AS AveragePrice,
    ROUND(CAST(MIN(Price) AS NUMERIC), 2) AS MinimumPrice,
    ROUND(CAST(MAX(Price) AS NUMERIC), 2) AS MaximumPrice
FROM Products;
```

### COUNT(*) vs. COUNT(column)


```sql
COUNT(*)
```

zählt alle Zeilen.


```sql
COUNT(CategoryID)
```

zählt nur Datensätze, bei denen `CategoryID` nicht `NULL` ist.

---

## 10. GROUP BY

`GROUP BY` fasst Zeilen anhand gemeinsamer Werte zusammen.

### Beispiel aus den Übungen

Anzahl Bestellungen pro Kunde:


```sql
SELECT 
    CustomerID, 
    COUNT(OrderID) AS OrderCount
FROM Orders
GROUP BY CustomerID
ORDER BY OrderCount DESC;
```

Logik:


```text
Orders
  -> nach CustomerID gruppieren
  -> Orders pro Gruppe zählen
  -> Ergebnis absteigend sortieren
```

---

## 11. HAVING

`WHERE` filtert **vor** der Gruppierung.  
`HAVING` filtert **nach** der Gruppierung.

### Beispiel aus den Übungen

Kunden mit mehr als drei Bestellungen:


```sql
SELECT 
    CustomerID, 
    COUNT(OrderID) AS OrderCount
FROM Orders
GROUP BY CustomerID
HAVING COUNT(OrderID) > 3;
```

### Merksatz


```text
WHERE  -> einzelne Zeilen
HAVING -> aggregierte Gruppen
```

---

## 12. INNER JOIN

Ein `INNER JOIN` liefert nur Datensätze, für die in beiden Tabellen ein passender Wert existiert.


```sql
SELECT
    c.CustomerName,
    o.OrderID
FROM Customers c
JOIN Orders o
    ON c.CustomerID = o.CustomerID;
```

### Beispiel mit drei Tabellen

Bestellpositionen für Kunden aus Frankreich:


```sql
SELECT 
    o.OrderID, 
    c.CustomerName, 
    c.Country, 
    od.ProductID, 
    od.Quantity
FROM Orders o
JOIN Customers c ON o.CustomerID = c.CustomerID
JOIN OrderDetails od ON o.OrderID = od.OrderID
WHERE c.Country = 'France';
```

Hier werden die Tabellen über ihre Schlüssel verbunden:


```text
Orders.CustomerID       -> Customers.CustomerID
Orders.OrderID          -> OrderDetails.OrderID
```

---

## 13. LEFT JOIN

Ein `LEFT JOIN` liefert **alle Datensätze der linken Tabelle**. Falls rechts kein passender Datensatz existiert, werden dort `NULL`-Werte erzeugt.


```sql
SELECT 
    o.OrderID, 
    e.EmployeeID, 
    e.FirstName, 
    e.LastName
FROM Orders o
LEFT JOIN Employees e
    ON o.EmployeeID = e.EmployeeID;
```

### Typischer Anwendungsfall

Alle Versandunternehmen anzeigen, auch wenn eines keine Bestellungen besitzt:


```sql
SELECT 
    s.ShipperName, 
    COUNT(o.OrderID) AS TotalOrders
FROM Shippers s
LEFT JOIN Orders o
    ON s.ShipperID = o.ShipperID
GROUP BY s.ShipperName;
```

### Wichtiger Pitfall

Eine Bedingung auf der rechten Tabelle in `WHERE` kann einen `LEFT JOIN` faktisch in einen `INNER JOIN` verwandeln.

Problematisch:


```sql
SELECT *
FROM Customers c
LEFT JOIN Orders o
    ON c.CustomerID = o.CustomerID
WHERE o.Status = 'Open';
```

Wenn Kunden ohne Order erhalten bleiben sollen, gehört die Bedingung häufig in `ON`:


```sql
SELECT *
FROM Customers c
LEFT JOIN Orders o
    ON c.CustomerID = o.CustomerID
   AND o.Status = 'Open';
```

---

## 14. Mehrere JOINs und Aggregation

### Beispiel aus den Übungen

Umsatz pro Produkt innerhalb einer Bestellung:


```sql
SELECT 
    od.OrderID,
    p.ProductID,
    p.ProductName,
    ROUND(CAST(SUM(p.Price * od.Quantity) AS NUMERIC), 2) AS TotalRevenue
FROM OrderDetails od
JOIN Products p ON od.ProductID = p.ProductID
GROUP BY od.OrderID, p.ProductID, p.ProductName
ORDER BY od.OrderID, TotalRevenue DESC;
```

Die Berechnung ist:


```text
TotalRevenue = Price * Quantity
```

und wird danach pro Bestellung und Produkt aggregiert.

---

## 15. Subquery

Eine **Subquery** ist eine SQL-Abfrage innerhalb einer anderen SQL-Abfrage.

### Beispiel aus den Übungen

Produkte, deren Preis über dem durchschnittlichen Produktpreis liegt:


```sql
SELECT 
    ProductID, 
    ProductName, 
    ROUND(CAST(Price AS NUMERIC), 2) AS Price
FROM Products
WHERE Price > (
    SELECT AVG(Price)
    FROM Products
)
ORDER BY Price DESC;
```

Ablauf:


```text
1. Durchschnittspreis berechnen
2. Produkte mit Price > Durchschnitt auswählen
3. Resultat absteigend sortieren
```

---

## 16. Logische SQL-Ausführungsreihenfolge

Eine SQL-Abfrage wird logisch ungefähr in dieser Reihenfolge verarbeitet:


```text
1. FROM
2. JOIN
3. WHERE
4. GROUP BY
5. HAVING
6. SELECT
7. DISTINCT
8. ORDER BY
9. LIMIT
```

Das erklärt beispielsweise, warum `WHERE` vor `GROUP BY` wirkt und `HAVING` danach.

---

## 17. SQL-Kurzschema


```sql
SELECT
    t.Column1,
    COUNT(*) AS Count
FROM TableName t
LEFT JOIN OtherTable o
    ON o.TableId = t.Id
WHERE t.Column1 IS NOT NULL
GROUP BY t.Column1
HAVING COUNT(*) > 1
ORDER BY Count DESC
LIMIT 10;
```

### Wichtigste Pitfalls Woche 1

- `NULL` mit `IS NULL` oder `IS NOT NULL` prüfen.
- `WHERE` und `HAVING` nicht verwechseln.
- Bei mehreren Tabellen Aliase verwenden.
- Bei `LEFT JOIN` Bedingungen auf der rechten Tabelle bewusst platzieren.
- Nicht aggregierte Spalten müssen bei `GROUP BY` normalerweise ebenfalls gruppiert werden.
- Ohne `ORDER BY` ist die Reihenfolge nicht garantiert.
- Ein Join kann die Anzahl der Zeilen vervielfachen; das ist bei 1:n-Beziehungen normal.